# Load clean CSVs into SQL Server

Loads the five files in `data/clean_data` into the `eu_trade` database on SQL Server Express,
one table per file, and checks that the row counts match.

**Prerequisites**
- SQL Server Express running, with the `eu_trade` database already created (`CREATE DATABASE eu_trade;`)
- ODBC Driver 18 for SQL Server installed (a Windows component, not a pip package)
- Project dependencies: `pip install -r requirements.txt`

Run the notebook from the project root. It should complete with Restart and Run All.

In [ ]:
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine

## Connection

`trusted_connection=yes` uses Windows authentication, `TrustServerCertificate=yes` accepts the
self-signed certificate of the local server, and `fast_executemany=True` inserts rows in batches
instead of one at a time.

The `r` prefix keeps `\S` from being read as an escape sequence.

In [ ]:
SERVER = r".\SQLEXPRESS"
DATABASE = "eu_trade"
CLEAN_DATA = Path("data/clean_data")

engine = create_engine(
    f"mssql+pyodbc://@{SERVER}/{DATABASE}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&trusted_connection=yes&TrustServerCertificate=yes",
    fast_executemany=True,
)

if not CLEAN_DATA.exists():
    raise FileNotFoundError(f"Folder not found: {CLEAN_DATA.resolve()}")

## Load

For each CSV: read it, check the column types, write it to SQL Server, compare row counts.

- `index=False` keeps the pandas index out of the table.
- `if_exists="replace"` drops and recreates the table, so the notebook can be re-run without
  duplicating rows.
- The `raise` stops the loop at the first mismatch: a printed `False` would go unnoticed in the
  middle of the output.

Check that year and value columns come back as `int64` or `float64`. A CSV is plain text, so pandas
infers the types on every read, and a single malformed cell turns a numeric column into text.

In [ ]:
for path in sorted(CLEAN_DATA.glob("*.csv")):
    table_name = path.stem.lower()
    print(f"\n{path.name} -> {table_name}")

    df = pd.read_csv(path)
    print(df.dtypes)

    df.to_sql(table_name, engine, index=False, if_exists="replace")

    sql_rows = pd.read_sql(f"SELECT COUNT(*) FROM {table_name}", engine).iloc[0, 0]
    if len(df) != sql_rows:
        raise ValueError(
            f"{table_name}: {len(df)} rows in the CSV, {sql_rows} in SQL Server"
        )

    print(f"OK: {len(df)} rows")

## Final check

Lists the loaded tables with their column count, reading the SQL Server metadata catalogue.

In [ ]:
query = """
SELECT TABLE_NAME, COUNT(*) AS n_columns
FROM INFORMATION_SCHEMA.COLUMNS
GROUP BY TABLE_NAME
ORDER BY TABLE_NAME
"""
pd.read_sql(query, engine)